In [ ]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "01_preprocessing"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


Mounted at /content/drive


In [ ]:
# Legacy Colab mount removido; paths definidos no setup.


BASE_PATH: /content/drive/MyDrive/TCC_USP
RAW_PATH : /content/drive/MyDrive/TCC_USP/data_raw
PROC_PATH: /content/drive/MyDrive/TCC_USP/data_processed


In [ ]:
# 3. Carregar IBOV
import pandas as pd, os
ibov_file = os.path.join(RAW_PATH, "ibovespa.csv")
assert os.path.exists(ibov_file), f"Arquivo nao encontrado: {ibov_file}"
df_ibov = pd.read_csv(ibov_file)

date_aliases = ["data", "day", "date"]
date_col = next((col for col in date_aliases if col in df_ibov.columns), None)
if date_col is None:
    raise KeyError("ibovespa.csv precisa de uma coluna de data (data/day/date)")
df_ibov["data"] = pd.to_datetime(df_ibov[date_col])
if date_col != "data":
    df_ibov = df_ibov.drop(columns=[date_col])

if "close" not in df_ibov.columns:
    if "adj_close" in df_ibov.columns:
        df_ibov["close"] = df_ibov["adj_close"]
    else:
        raise KeyError("ibovespa.csv precisa da coluna close ou adj_close")

print("IBOV:", df_ibov.shape)
display(df_ibov.head())


IBOV: (20, 2)


,data,close
0,2024-01-02,130749.73
1,2024-01-03,130673.55
2,2024-01-04,131624.44
3,2024-01-05,133734.42
4,2024-01-08,133528.27


In [ ]:
# 4. Carregar NotÃ­cias
news_file = os.path.join(RAW_PATH, "noticias_exemplo.csv")
assert os.path.exists(news_file), f"Arquivo nÃ£o encontrado: {news_file}"
df_news = pd.read_csv(news_file)
df_news["data"] = pd.to_datetime(df_news["data"])
print("NEWS:", df_news.shape)
display(df_news.head())

NEWS: (10, 5)


,data,fonte,titulo,texto,sentimento
0,2024-01-02,exemplo,Titulo 1,Ibovespa sobe com otimismo no mercado internac...,1
1,2024-01-03,exemplo,Titulo 2,Bolsa cai apÃ³s dados fracos da economia chinesa.,0
2,2024-01-04,exemplo,Titulo 3,PetrÃ³leo em alta puxa aÃ§Ãµes da Petrobras para ...,1
3,2024-01-05,exemplo,Titulo 4,Queda do dÃ³lar beneficia empresas exportadoras.,1
4,2024-01-08,exemplo,Titulo 5,Setor bancÃ¡rio recua com aversÃ£o ao risco.,0


In [ ]:
# 5. Pré-processamento de texto
import re, nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_pt = set(stopwords.words("portuguese"))
PATTERN = r"[^A-Za-z\u00C0-\u00FF\s]"

def preprocess_text(t):
    t = re.sub(PATTERN, " ", str(t)).lower()
    toks = [w for w in t.split() if w and w not in stop_pt]
    return " ".join(toks)

# Usar a coluna 'texto'
df_news["clean_text"] = df_news["texto"].apply(preprocess_text)

print("✅ Pré-processamento concluído!")
display(df_news[["texto", "clean_text"]].head())


âœ… PrÃ©-processamento concluÃ­do!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,texto,clean_text
0,Ibovespa sobe com otimismo no mercado internac...,ibovespa sobe otimismo mercado internacional
1,Bolsa cai apÃ³s dados fracos da economia chinesa.,bolsa cai apÃ³s dados fracos economia chinesa
2,PetrÃ³leo em alta puxa aÃ§Ãµes da Petrobras para ...,petrÃ³leo alta puxa aÃ§Ãµes petrobras cima
3,Queda do dÃ³lar beneficia empresas exportadoras.,queda dÃ³lar beneficia empresas exportadoras
4,Setor bancÃ¡rio recua com aversÃ£o ao risco.,setor bancÃ¡rio recua aversÃ£o risco


In [38]:
from pathlib import Path
print("Arquivos em PROC_PATH:")
for path in Path(PROC_PATH).iterdir():
    size_kb = path.stat().st_size / 1024
    print(f" - {path.name} ({size_kb:0.1f} KB)")

Salvo em data_processed:
total 2.0K
-rw------- 1 root root  427 Sep 20 21:47 ibovespa_clean.csv
-rw------- 1 root root 1.3K Sep 20 21:47 noticias_clean.csv
